In [1]:
!pip install smplx trimesh yacs chumpy -q
!sed -i "s/order is 'C'/order == 'C'/g" /usr/local/lib/python3.12/dist-packages/chumpy/ch_ops.py
!git clone --recurse-submodules -j4 https://github.com/filby89/spectre -q
!pip install pyrender moviepy imageio[ffmpeg] -q
!apt-get install -y libegl1-mesa libegl1-mesa-dev -qq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.6/50.6 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 39.5 MB/s eta 0:00:00
Error downloading object: ibug/face_detection/retina_face/weights/Resnet50_Final.pth (6d1de9c): Smudge error: Error downloading ibug/face_detection/retina_face/weights/Resnet50_Final.pth (6d1de9c2944f2ccddca5f5e010ea5ae64a39845a86311af6fdf30841b0a5a16d): batch response: This repository exceeded its LFS budget. The account responsible for the budget should increase it to restore access.

Errors logged to /kaggle/working/spectre/.git/modules/external/face_detection/lfs/logs/20260520T003401.900283781.log
Use `git lfs logs last` to view the log.
error: external filter 'git-lfs filter-process' failed
fatal: ibug/face_detection/retina_face/weights/Resnet50_Final.pth: smudge filter lfs failed
fatal: Unable to checkout '71852f00b815f568f3b51f045a418ae84cbe162a' in submodule path 'external/face_d

In [2]:
import os
import sys
import math
import glob
import time
import torch
import inspect
import trimesh
import imageio
import pyrender
import numpy as np
from tqdm import tqdm
from enum import Enum
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.utils.rnn as rnn_utils
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import WeightedRandomSampler
from moviepy.editor import AudioFileClip, VideoFileClip

os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if not hasattr(inspect, 'getargspec'):
    inspect.getargspec = inspect.getfullargspec

sys.path.append('/kaggle/working/spectre')
from config import cfg as spectre_cfg
from src.models.FLAME import FLAME

np.bool    = np.bool_
np.int     = np.int64
np.float   = np.float64
np.complex = np.complex128
np.object  = np.object_
np.unicode = np.str_
np.str     = np.str_

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} | GPUs: {torch.cuda.device_count()}")

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
error: XDG_RUNTIME_DIR not set in the environment.
ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
A

Device: cuda | GPUs: 2


In [3]:
flame_model_path = '/kaggle/input/datasets/jorx04/flame-2020-weights/FLAME2020/generic_model.pkl'
spectre_cfg.defrost()
spectre_cfg.model.flame_model_path = flame_model_path
spectre_cfg.model.use_tex = False
spectre_cfg.freeze()

flame_model = FLAME(spectre_cfg.model).to(device)

def fix_module_tensors(module):
    for attr_name, attr_value in list(vars(module).items()):
        if isinstance(attr_value, torch.Tensor) and not isinstance(attr_value, nn.Parameter):
            delattr(module, attr_name)
            module.register_buffer(attr_name, attr_value)
    for child in module.children():
        fix_module_tensors(child)

fix_module_tensors(flame_model)
flame_model.eval()
print("FLAME loaded.")

class TrainingMode(Enum):
    NORMAL       = 1
    OVERFIT_TEST = 2

ACTIVE_MODE = TrainingMode.NORMAL

FLAME loaded.


In [4]:
class FLAMEOnlyDataset(Dataset):
    def __init__(self, flame_dirs, mode=TrainingMode.NORMAL):
        self.mode    = mode
        self.samples = []
        for f_dir in flame_dirs:
            for f_path in glob.glob(os.path.join(f_dir, '*.pt')):
                self.samples.append(f_path)
        if mode == TrainingMode.OVERFIT_TEST and len(self.samples) > 0:
            self.samples = [self.samples[0]]
        print(f'FLAMEOnlyDataset: {len(self.samples)} sequences found.')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        f_path      = self.samples[idx]
        flame_dict  = torch.load(f_path, weights_only=True)
        expressions = flame_dict['exp']
        jaw_pose    = flame_dict['pose'][:, 3:6]
        shape_param = flame_dict['shape'][0:1]
        flame_params = torch.cat([expressions, jaw_pose], dim=-1)
        return flame_params[:, :53], shape_param


def flame_collate_fn(batch):
    flames = [item[0] for item in batch]
    shapes = [item[1] for item in batch]
    lengths = torch.tensor([f.shape[0] for f in flames], dtype=torch.long)
    max_len = lengths.max()
    mask    = torch.arange(max_len).expand(len(lengths), max_len) < lengths.unsqueeze(1)
    flames_padded  = rnn_utils.pad_sequence(flames, batch_first=True)
    shapes_batched = torch.stack(shapes, dim=0)
    return {'flame': flames_padded, 'shape': shapes_batched, 'mask': mask}

In [5]:
class RotaryEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq)
        
    def forward(self, seq_len, device):
        t = torch.arange(seq_len, device=device).type_as(self.inv_freq)
        freqs = torch.outer(t, self.inv_freq)
        return torch.cat((freqs, freqs), dim=-1)

def apply_rotary_emb(x, freqs):
    d = x.shape[-1] // 2
    x1, x2 = x[..., :d], x[..., d:]
    x_rot = torch.cat((-x2, x1), dim=-1)
    return x * freqs.cos() + x_rot * freqs.sin()

class SDPATransformerLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, dropout):
        super().__init__()
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(dim_feedforward, d_model)
        )
        self.nhead = nhead

    def forward(self, x, rope_freqs, mask=None):
        B, T, C = x.shape
        H = self.nhead
        D = C // H
        
        q, k, v = self.qkv(self.norm1(x)).chunk(3, dim=-1)
        q, k, v = q.view(B, T, H, D), k.view(B, T, H, D), v.view(B, T, H, D)
        
        q = apply_rotary_emb(q, rope_freqs.unsqueeze(0).unsqueeze(2))
        k = apply_rotary_emb(k, rope_freqs.unsqueeze(0).unsqueeze(2))
        
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
        
        attn_mask = mask.unsqueeze(1).unsqueeze(2) if mask is not None else None
        
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=0.0 if not self.training else 0.1)
        x = x + self.proj(out.transpose(1, 2).reshape(B, T, C))
        x = x + self.ffn(self.norm2(x))
        return x

class FLAMEKLEncoder(nn.Module):
    def __init__(self, flame_dim=53, hidden_dim=256, latent_dim=32, num_layers=4, num_heads=4, dropout=0.1, compression=2):
        super().__init__()
        self.compression = compression
        self.input_proj  = nn.Linear(flame_dim, hidden_dim)
        self.rope = RotaryEmbedding(hidden_dim // num_heads)
        self.layers = nn.ModuleList([SDPATransformerLayer(hidden_dim, num_heads, hidden_dim * 4, dropout) for _ in range(num_layers)])
        self.temporal_pool = nn.Conv1d(hidden_dim, hidden_dim, kernel_size=2, stride=2)
        self.to_mean    = nn.Linear(hidden_dim, latent_dim)
        self.to_logvar  = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x, mask=None):
        h = self.input_proj(x)
        freqs = self.rope(h.shape[1], h.device)
        for layer in self.layers:
            h = layer(h, freqs, mask)
        if self.compression > 1:
            h = self.temporal_pool(h.transpose(1, 2)).transpose(1, 2)
        return self.to_mean(h), self.to_logvar(h)

class FLAMEKLDecoder(nn.Module):
    def __init__(self, flame_dim=53, hidden_dim=256, latent_dim=32, num_layers=4, num_heads=4, dropout=0.1, compression=2):
        super().__init__()
        self.compression = compression
        self.input_proj  = nn.Linear(latent_dim, hidden_dim)
        self.temporal_upsample = nn.ConvTranspose1d(hidden_dim, hidden_dim, kernel_size=2, stride=2)
        self.rope = RotaryEmbedding(hidden_dim // num_heads)
        self.layers = nn.ModuleList([SDPATransformerLayer(hidden_dim, num_heads, hidden_dim * 4, dropout) for _ in range(num_layers)])
        self.out_proj    = nn.Linear(hidden_dim, flame_dim)

    def forward(self, z, mask=None):
        h = self.input_proj(z)
        if self.compression > 1:
            h = self.temporal_upsample(h.transpose(1, 2)).transpose(1, 2)
        
        freqs = self.rope(h.shape[1], h.device)
        aligned_mask = mask[:, :h.shape[1]] if mask is not None else None
        
        for layer in self.layers:
            h = layer(h, freqs, aligned_mask)
        return self.out_proj(h)

class FLAMEKLVAE(nn.Module):
    def __init__(self, flame_dim=53, hidden_dim=256, latent_dim=32, num_layers=4, num_heads=4, dropout=0.1, compression=2):
        super().__init__()
        self.encoder = FLAMEKLEncoder(flame_dim, hidden_dim, latent_dim, num_layers, num_heads, dropout, compression)
        self.decoder = FLAMEKLDecoder(flame_dim, hidden_dim, latent_dim, num_layers, num_heads, dropout, compression)
        self.latent_dim  = latent_dim
        self.compression = compression

    def reparameterise(self, mean, logvar):
        if self.training:
            std = (0.5 * logvar).exp()
            return mean + std * torch.randn_like(std)
        return mean

    def encode(self, x, mask=None):
        B, T, D = x.shape
        pad = (self.compression - T % self.compression) % self.compression
        if pad > 0:
            x    = F.pad(x, (0, 0, 0, pad))
            mask = F.pad(mask, (0, pad), value=False) if mask is not None else None
        mean, _ = self.encoder(x, mask)
        return mean

    def forward(self, x, mask=None):
        B, T, D = x.shape

        pad = (self.compression - T % self.compression) % self.compression
        if pad > 0:
            x    = F.pad(x, (0, 0, 0, pad))
            mask = F.pad(mask, (0, pad), value=False) if mask is not None else None

        mean, logvar = self.encoder(x, mask)
        logvar = logvar.clamp(-30.0, 20.0)
        z      = self.reparameterise(mean, logvar)
        recon  = self.decoder(z, mask)

        recon = recon[:, :T, :]
        return recon, mean, logvar

In [6]:
raw_mouth = [1576, 1577, 1578, 1579, 1583, 1657, 1667, 1668, 1669, 1670, 1691, 1693, 1694, 1695, 1696, 1697, 1700, 1702, 1703, 1704, 1709, 1710, 1711, 1712, 1713, 1714, 1715, 1716, 1717, 1719, 1720, 1721, 1731, 1732, 1733, 1734, 1735, 1736, 1737, 1738, 1740, 1748, 1749, 1750, 1751, 1758, 1763, 1770, 1771, 1773, 1774, 1775, 1776, 1777, 1778, 1787, 1788, 1789, 1791, 1792, 1793, 1794, 1795, 1796, 1801, 1802, 1803, 1804, 1826, 1827, 1836, 1848, 1849, 1850, 1865, 1866, 2712, 2713, 2714, 2715, 2716, 2719, 2774, 2784, 2785, 2786, 2787, 2811, 2812, 2813, 2814, 2817, 2819, 2820, 2821, 2826, 2827, 2828, 2829, 2830, 2831, 2832, 2833, 2834, 2836, 2837, 2838, 2840, 2845, 2846, 2847, 2848, 2849, 2850, 2851, 2852, 2853, 2855, 2863, 2864, 2865, 2866, 2869, 2871, 2878, 2879, 2880, 2881, 2882, 2883, 2884, 2885, 2890, 2891, 2892, 2894, 2895, 2896, 2897, 2898, 2899, 2904, 2905, 2906, 2907, 2928, 2929, 2934, 2937, 2938, 2939, 2948, 2949, 3503, 3504, 3506, 3509, 3511, 3513, 3531, 3533, 3543, 3546, 3790, 3791, 3792, 3793, 3795, 3796, 3797, 3798, 3799, 3800, 3805, 3806, 3914, 3915, 3916, 3917, 3918, 3919, 3920, 3921, 3922, 3923, 3927, 3928]
raw_lower = [63, 64, 123, 124, 143, 174, 177, 183, 184, 223, 224, 245, 250, 270, 271, 365, 532, 560, 561, 562, 565, 566, 567, 568, 613, 726, 729, 784, 785, 856, 859, 867, 868, 933, 934, 1009, 1049, 1050, 1087, 1092, 1479, 1480, 1496, 1530, 1531, 1569, 1570, 1574, 1575, 1576, 1577, 1578, 1579, 1584, 1585, 1586, 1587, 1596, 1597, 1598, 1599, 1601, 1602, 1657, 1658, 1669, 1670, 1686, 1687, 1688, 1689, 1690, 1691, 1693, 1694, 1695, 1696, 1697, 1698, 1699, 1700, 1701, 1702, 1703, 1704, 1705, 1706, 1707, 1708, 1709, 1710, 1711, 1712, 1713, 1714, 1715, 1716, 1717, 1731, 1732, 1733, 1734, 1735, 1736, 1737, 1738, 1749, 1750, 1751, 1758, 1763, 1766, 1767, 1768, 1769, 1770, 1771, 1773, 1774, 1775, 1776, 1791, 1792, 1793, 1794, 1795, 1796, 1797, 1798, 1799, 1800, 1801, 1802, 1803, 1804, 1826, 1848, 1849, 1850, 1853, 1863, 1864, 1865, 1866, 1935, 1936, 1937, 1960, 1961, 1962, 1963, 1998, 1999, 2004, 2009, 2011, 2012, 2078, 2079, 2080, 2081, 2083, 2175, 2185, 2208, 2211, 2252, 2399, 2401, 2616, 2617, 2666, 2667, 2705, 2706, 2710, 2711, 2712, 2713, 2714, 2715, 2720, 2721, 2722, 2723, 2732, 2733, 2734, 2735, 2737, 2738, 2774, 2775, 2786, 2787, 2803, 2804, 2805, 2806, 2807, 2808, 2810, 2811, 2812, 2813, 2814, 2815, 2816, 2817, 2818, 2819, 2820, 2821, 2822, 2823, 2824, 2825, 2826, 2827, 2828, 2829, 2830, 2831, 2832, 2833, 2834, 2847, 2848, 2849, 2850, 2851, 2852, 2853, 2864, 2865, 2866, 2871, 2874, 2875, 2876, 2877, 2878, 2879, 2880, 2881, 2882, 2883, 2890, 2891, 2894, 2895, 2896, 2897, 2898, 2899, 2900, 2901, 2902, 2903, 2904, 2905, 2906, 2907, 2928, 2937, 2938, 2939, 2946, 2947, 2948, 2949, 3054, 3055, 3056, 3057, 3059, 3060, 3112, 3113, 3114, 3115, 3116, 3118, 3127, 3183, 3381, 3382, 3383, 3384, 3385, 3386, 3387, 3388, 3389, 3390, 3391, 3392, 3393, 3394, 3395, 3396, 3397, 3398, 3399, 3400, 3401, 3402, 3403, 3404, 3405, 3406, 3407, 3408, 3409, 3410, 3411, 3412, 3413, 3414, 3415, 3416, 3417, 3418, 3419, 3420, 3421, 3422, 3423, 3424, 3425, 3426, 3427, 3428, 3429, 3430, 3431, 3432, 3433, 3434, 3435, 3436, 3437, 3438, 3439, 3440, 3441, 3442, 3443, 3444, 3445, 3446, 3447, 3448, 3449, 3450, 3451, 3453, 3454, 3455, 3456, 3457, 3458, 3459, 3464, 3465, 3466, 3467, 3468, 3469, 3470, 3471, 3472, 3473, 3474, 3475, 3476, 3477, 3478, 3479, 3480, 3481, 3482, 3483, 3484, 3485, 3486, 3487, 3489, 3490, 3491, 3492, 3493, 3499, 3502, 3503, 3504, 3506, 3509, 3511, 3515, 3531, 3537, 3538, 3541, 3543, 3546, 3577, 3578, 3579, 3580, 3581, 3582, 3583, 3584, 3587, 3588, 3593, 3594, 3595, 3596, 3598, 3599, 3600, 3601, 3604, 3605, 3611, 3614, 3623, 3624, 3625, 3626, 3628, 3629, 3630, 3634, 3635, 3636, 3637, 3643, 3644, 3646, 3649, 3650, 3652, 3653, 3654, 3655, 3656, 3658, 3659, 3660, 3662, 3663, 3664, 3665, 3666, 3667, 3670, 3671, 3672, 3673, 3676, 3677, 3678, 3679, 3680, 3681, 3685, 3691, 3693, 3697, 3698, 3701, 3703, 3707, 3714, 3715, 3716, 3717, 3722, 3724, 3725, 3726, 3727, 3728, 3730, 3734, 3737, 3738, 3739, 3740, 3742, 3745, 3752, 3753, 3754, 3756, 3757, 3761, 3762, 3769, 3771, 3772, 3790, 3791, 3792, 3793, 3794, 3795, 3796, 3797, 3798, 3799, 3800, 3801, 3802, 3803, 3804, 3805, 3806, 3914, 3915, 3916, 3917, 3918, 3919, 3920, 3921, 3922, 3923, 3924, 3925, 3926, 3927, 3928]
raw_upper = [16, 17, 18, 27, 335, 336, 337, 338, 498, 499, 500, 501, 569, 570, 571, 572, 589, 590, 591, 592, 605, 622, 623, 624, 625, 626, 627, 628, 629, 630, 667, 668, 669, 670, 671, 672, 673, 674, 679, 680, 681, 682, 683, 688, 693, 694, 695, 696, 697, 702, 703, 704, 705, 706, 707, 708, 713, 714, 715, 723, 724, 725, 738, 739, 807, 808, 809, 814, 815, 816, 821, 822, 823, 824, 825, 827, 828, 829, 841, 842, 848, 864, 865, 877, 878, 879, 880, 881, 882, 883, 884, 885, 896, 897, 898, 899, 902, 903, 904, 905, 906, 907, 908, 909, 918, 919, 922, 923, 924, 926, 927, 928, 929, 939, 942, 943, 944, 945, 946, 947, 948, 949, 950, 951, 952, 953, 954, 955, 958, 959, 960, 961, 962, 963, 964, 965, 966, 967, 968, 969, 970, 971, 972, 977, 978, 979, 980, 985, 986, 991, 994, 995, 999, 1012, 1013, 1014, 1015, 1019, 1020, 1021, 1022, 1023, 1033, 1034, 1043, 1044, 1059, 1060, 1062, 1088, 1093, 1096, 1101, 1108, 1113, 1114, 1115, 1125, 1132, 1134, 1135, 1142, 1143, 1144, 1146, 1147, 1151, 1152, 1153, 1154, 1155, 1161, 1162, 1163, 1164, 1168, 1169, 1170, 1175, 1176, 1181, 1182, 1183, 1184, 1189, 1190, 1193, 1194, 1195, 1200, 1201, 1202, 1216, 1218, 1225, 1226, 1232, 1233, 1243, 1244, 1292, 1293, 1294, 1329, 1331, 1336, 1337, 1338, 1339, 1340, 1341, 1342, 1343, 1344, 1345, 1346, 1347, 1348, 1349, 1350, 1351, 1352, 1353, 1354, 1355, 1356, 1357, 1358, 1361, 1362, 1363, 1364, 1365, 1366, 1367, 1368, 1369, 1370, 1371, 1372, 1373, 1374, 1375, 1376, 1377, 1378, 1383, 1384, 1385, 1387, 1388, 1389, 1390, 1391, 1396, 1397, 1398, 1399, 1400, 1401, 1402, 1403, 1404, 1405, 1410, 1411, 1412, 1413, 1414, 1415, 1416, 1417, 1418, 1419, 1420, 1421, 1422, 1425, 1426, 1427, 1428, 1429, 1430, 1431, 1432, 1433, 1434, 1435, 1436, 1437, 1438, 1439, 1440, 1441, 1442, 1443, 1444, 1445, 1446, 1447, 1448, 1449, 1450, 1451, 1452, 1453, 1458, 1459, 1460, 1461, 1462, 1463, 1464, 1465, 1466, 1467, 1468, 1469, 1473, 1474, 1475, 1476, 1477, 1478, 1641, 1642, 1643, 1644, 1649, 1752, 1753, 1754, 1755, 1982, 1983, 1984, 1985, 2017, 2018, 2019, 2020, 2030, 2043, 2044, 2045, 2046, 2047, 2048, 2049, 2050, 2051, 2053, 2054, 2058, 2059, 2060, 2061, 2062, 2063, 2069, 2070, 2071, 2072, 2073, 2074, 2075, 2076, 2082, 2084, 2085, 2086, 2087, 2088, 2089, 2090, 2091, 2092, 2093, 2094, 2095, 2096, 2097, 2098, 2099, 2100, 2101, 2102, 2103, 2104, 2105, 2106, 2107, 2108, 2109, 2110, 2111, 2112, 2113, 2114, 2118, 2119, 2120, 2121, 2122, 2123, 2124, 2125, 2126, 2127, 2134, 2135, 2138, 2139, 2140, 2165, 2166, 2167, 2168, 2176, 2177, 2178, 2179, 2180, 2181, 2182, 2183, 2184, 2185, 2188, 2189, 2190, 2191, 2192, 2193, 2194, 2195, 2196, 2197, 2198, 2199, 2202, 2203, 2204, 2205, 2206, 2207, 2220, 2221, 2264, 2265, 2266, 2267, 2268, 2269, 2270, 2271, 2272, 2273, 2274, 2276, 2277, 2278, 2282, 2283, 2286, 2287, 2288, 2289, 2290, 2291, 2292, 2293, 2294, 2295, 2296, 2297, 2298, 2299, 2300, 2301, 2302, 2303, 2304, 2305, 2306, 2307, 2308, 2309, 2310, 2311, 2312, 2313, 2314, 2315, 2316, 2317, 2318, 2319, 2320, 2321, 2322, 2323, 2324, 2325, 2326, 2327, 2328, 2329, 2330, 2331, 2332, 2333, 2334, 2335, 2336, 2337, 2338, 2339, 2340, 2341, 2342, 2343, 2344, 2345, 2346, 2347, 2348, 2349, 2350, 2351, 2352, 2353, 2354, 2358, 2359, 2360, 2370, 2371, 2372, 2373, 2377, 2378, 2379, 2380, 2381, 2382, 2383, 2384, 2385, 2388, 2389, 2391, 2400, 2402, 2403, 2404, 2405, 2406, 2407, 2408, 2411, 2416, 2417, 2418, 2419, 2420, 2421, 2422, 2423, 2425, 2426, 2427, 2428, 2429, 2430, 2431, 2432, 2433, 2434, 2435, 2436, 2437, 2438, 2439, 2440, 2441, 2442, 2443, 2444, 2445, 2446, 2447, 2448, 2449, 2450, 2451, 2452, 2453, 2454, 2455, 2456, 2461, 2462, 2465, 2466, 2471, 2472, 2473, 2485, 2486, 2487, 2488, 2489, 2490, 2491, 2492, 2493, 2494, 2495, 2496, 2497, 2498, 2499, 2500, 2501, 2502, 2503, 2504, 2505, 2506, 2507, 2508, 2509, 2510, 2511, 2512, 2513, 2514, 2515, 2516, 2517, 2518, 2519, 2520, 2521, 2522, 2523, 2524, 2525, 2526, 2527, 2528, 2529, 2530, 2532, 2533, 2534, 2535, 2536, 2537, 2538, 2539, 2540, 2541, 2542, 2543, 2544, 2545, 2546, 2547, 2548, 2549, 2550, 2551, 2552, 2553, 2554, 2555, 2556, 2557, 2558, 2559, 2562, 2563, 2564, 2565, 2566, 2567, 2568, 2569, 2570, 2571, 2572, 2573, 2574, 2575, 2576, 2577, 2578, 2579, 2580, 2581, 2582, 2583, 2584, 2585, 2586, 2587, 2588, 2589, 2590, 2595, 2596, 2597, 2598, 2599, 2600, 2601, 2602, 2603, 2604, 2605, 2606, 2610, 2611, 2612, 2613, 2614, 2615, 2741, 2742, 2758, 2759, 2760, 2761, 2766, 3068, 3078, 3079, 3080, 3081, 3082, 3083, 3084, 3085, 3086, 3088, 3089, 3090, 3093, 3094, 3095, 3096, 3097, 3098, 3104, 3105, 3106, 3107, 3108, 3109, 3110, 3111, 3117, 3119, 3120, 3121, 3122, 3123, 3124, 3125, 3126, 3127, 3128, 3129, 3130, 3131, 3132, 3133, 3134, 3135, 3136, 3137, 3138, 3139, 3140, 3141, 3145, 3146, 3147, 3148, 3149, 3150, 3151, 3152, 3153, 3154, 3157, 3158, 3159, 3495, 3516, 3518, 3524, 3540, 3542, 3550, 3553, 3555, 3560, 3572, 3589, 3592, 3603, 3615, 3619, 3620, 3622, 3631, 3632, 3638, 3645, 3647, 3648, 3674, 3675, 3682, 3683, 3684, 3686, 3687, 3688, 3689, 3690, 3692, 3694, 3696, 3699, 3704, 3705, 3706, 3708, 3710, 3711, 3712, 3718, 3720, 3721, 3729, 3731, 3732, 3733, 3735, 3741, 3743, 3744, 3747, 3749, 3750, 3759, 3763, 3764, 3766, 3767, 3770, 3773, 3774, 3775, 3776, 3777, 3779, 3780, 3781, 3784, 3786, 3787, 3788, 3789, 3809, 3812, 3815, 3823, 3827, 3828, 3830, 3832, 3833, 3835, 3840, 3841, 3842, 3847, 3848, 3849, 3850, 3851, 3852, 3854, 3855, 3856, 3857, 3858, 3859, 3860, 3863, 3864, 3865, 3866, 3867, 3868, 3869, 3871, 3872, 3874, 3875, 3876, 3877, 3878, 3880, 3881, 3882, 3884, 3886, 3887, 3891, 3892, 3893, 3895, 3896, 3898, 3899, 3900, 3901, 3902, 3903, 3905, 3906, 3907, 3910, 3911, 3912, 3913, 3965, 4021, 4041, 4271, 4556, 4563, 4609]

mouth_set = set(raw_mouth)
lower_set = set(raw_lower) - mouth_set
upper_set = set(raw_upper) - mouth_set - lower_set

clean_mouth = sorted(list(mouth_set))
clean_lower = sorted(list(lower_set))
clean_upper = sorted(list(upper_set))

print(f"Final counts -> Mouth: {len(clean_mouth)}, Lower: {len(clean_lower)}, Upper: {len(clean_upper)}")

Final counts -> Mouth: 187, Lower: 384, Upper: 967


In [7]:
class DynamicLossWeights:
    def __init__(self, total_epochs=200, t0=100):
        self.total_epochs = total_epochs
        self.t0 = t0

    def get_weights(self, epoch):
        total = self.total_epochs
        t0    = self.t0
        prog  = epoch / total

        w_recon  = 5.0 - 2.0 * prog
        w_motion = 0.1 + 0.9 * min((prog * 2), 1.0)
        w_jaw    = 5.0 + 95.0 * prog

        base_mouth = 1000.0 + 1000.0 * prog
        base_lower =  500.0 +  500.0 * prog
        in_restart_dip = t0 <= epoch < t0 + int(0.15 * total)
        dip = 0.6 if in_restart_dip else 1.0
        w_mouth = base_mouth * dip
        w_lower = base_lower * dip

        w_upper = 1500.0 - 500.0 * prog

        return dict(
            w_recon=w_recon, w_motion=w_motion, w_jaw=w_jaw,
            w_mouth=w_mouth, w_lower=w_lower, w_upper=w_upper
        )


class VertexReconLoss(nn.Module):
    def __init__(self, flame_model, device):
        super().__init__()
        self.flame  = flame_model
        self.device = device

        self.w_recon  = 5.0
        self.w_motion = 0.1
        self.w_jaw    = 5.0
        self.w_mouth  = 1000.0
        self.w_lower  =  500.0
        self.w_upper  = 1500.0

        raw_mouth = torch.tensor(clean_mouth, dtype=torch.long)
        raw_lower = torch.tensor(clean_lower, dtype=torch.long)
        raw_upper = torch.tensor(clean_upper, dtype=torch.long)
        self.register_buffer('mouth_indices', raw_mouth)
        self.register_buffer('lower_indices', raw_lower)
        self.register_buffer('upper_indices', raw_upper)

    def update_weights(self, weights: dict):
        for k, v in weights.items():
            setattr(self, k, v)

    def _build_weight_map(self, device):
        w = torch.ones(5023, device=device)
        w[self.lower_indices] = self.w_lower
        w[self.upper_indices] = self.w_upper
        w[self.mouth_indices] = self.w_mouth
        return w.view(1, 1, 5023, 1)

    def _to_verts(self, params_flat, shape_exp):
        current_device = params_flat.device
        exp  = params_flat[:, :50]
        pose = torch.zeros(params_flat.shape[0], 6,
                           device=current_device, dtype=params_flat.dtype)
        pose[:, 3:6] = params_flat[:, 50:53]
        return self.flame(
            shape_params=shape_exp,
            expression_params=exp,
            pose_params=pose
        )[0]

    def forward(self, pred, gt, mask, gt_shape):
        B, T, D = pred.shape
        if gt_shape.dim() == 3:
            gt_shape = gt_shape.squeeze(1)
        shape_exp = gt_shape.unsqueeze(1).expand(B, T, 100).reshape(B * T, 100)

        pred_v = self._to_verts(pred.reshape(B*T, D), shape_exp).reshape(B, T, 5023, 3)
        gt_v   = self._to_verts(gt.reshape(B*T, D),   shape_exp).reshape(B, T, 5023, 3)

        w  = self._build_weight_map(pred.device)
        m4 = mask.unsqueeze(-1).unsqueeze(-1).expand_as(pred_v).float()

        loss_recon = ((pred_v - gt_v).pow(2) * w * m4).sum() / (m4.sum() + 1e-8)

        loss_motion = torch.tensor(0.0, device=pred.device)
        if T > 2:
            if not hasattr(self, '_w_smooth') or self._w_smooth.device != pred.device:
                ws = self._build_weight_map(pred.device).clone()
                ws[0, 0, self.upper_indices, :] /= 2.0
                self.register_buffer('_w_smooth', ws)
            accel = pred_v[:, 2:] - 2 * pred_v[:, 1:-1] + pred_v[:, :-2]
            mm    = m4[:, 2:]
            loss_motion = (accel.pow(2) * self._w_smooth * mm).sum() / (mm.sum() + 1e-8)

        jaw_err  = (pred[:, :, 50:53] - gt[:, :, 50:53]).pow(2)
        mj       = mask.unsqueeze(-1).expand_as(jaw_err).float()
        loss_jaw = (jaw_err * mj).sum() / (mj.sum() + 1e-8)

        return (self.w_recon  * loss_recon
              + self.w_motion * loss_motion
              + self.w_jaw    * loss_jaw)


In [8]:
LATENT_DIM           = 32
HIDDEN_DIM           = 256
NUM_LAYERS           = 4
NUM_HEADS            = 4
BATCH_SIZE           = 16
LR                   = 1e-4
EPOCHS               = 200
VAL_SPLIT            = 0.1
SEED                 = 0
TEMPORAL_COMPRESSION = 2

KL_BETA_TARGET   = 1e-4
KL_WARMUP_EPOCHS = 40

checkpoint_path = None

klvae = FLAMEKLVAE(
    flame_dim=53, hidden_dim=HIDDEN_DIM, latent_dim=LATENT_DIM,
    num_layers=NUM_LAYERS, num_heads=NUM_HEADS, dropout=0.15,
    compression=TEMPORAL_COMPRESSION
).to(device)

criterion = VertexReconLoss(flame_model=flame_model, device=device).to(device)
optimizer = torch.optim.AdamW(klvae.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=100, T_mult=1, eta_min=1e-6
)

loss_scheduler = DynamicLossWeights(total_epochs=EPOCHS, t0=100)

start_epoch   = 0
best_val_loss = float('inf')

if checkpoint_path and os.path.exists(checkpoint_path):
    ckpt = torch.load(checkpoint_path, map_location=device)
    klvae.load_state_dict(ckpt['klvae_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    start_epoch   = ckpt['epoch']
    best_val_loss = ckpt.get('best_val_loss', float('inf'))
    print(f'Resumed from epoch {start_epoch}, best val loss {best_val_loss:.4f}')
else:
    print('Starting from scratch.')

if torch.cuda.device_count() > 1:
    print(f'Using {torch.cuda.device_count()} GPUs')
    klvae       = nn.DataParallel(klvae)
    criterion   = nn.DataParallel(criterion)
    flame_model = nn.DataParallel(flame_model)

flame_dirs = [
    '/kaggle/input/datasets/jorx04/fully-collected-dataset/flame/ara',
    '/kaggle/input/datasets/jorx04/fully-collected-dataset/flame/eng',
]

full_dataset = FLAMEOnlyDataset(flame_dirs, mode=ACTIVE_MODE)
torch.manual_seed(SEED)
n_total = len(full_dataset)
n_val   = max(1, int(n_total * VAL_SPLIT))
n_train = n_total - n_val
train_dataset, val_dataset = torch.utils.data.random_split(
    full_dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED)
)

train_weights = []
weight_eng = 1.0 / 11000.0
weight_ara = 1.0 / 8000.0

for idx in train_dataset.indices:
    f_path = full_dataset.samples[idx]
    if '/ara/' in f_path:
        train_weights.append(weight_ara)
    else:
        train_weights.append(weight_eng)

train_sampler = WeightedRandomSampler(
    weights=train_weights,
    num_samples=len(train_weights),
    replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE if ACTIVE_MODE == TrainingMode.NORMAL else 1,
    sampler=train_sampler if ACTIVE_MODE == TrainingMode.NORMAL else None,
    shuffle=False,
    collate_fn=flame_collate_fn,
    num_workers=4, pin_memory=True, prefetch_factor=2, persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE if ACTIVE_MODE == TrainingMode.NORMAL else 1,
    shuffle=False, collate_fn=flame_collate_fn,
    num_workers=2, pin_memory=True, prefetch_factor=2, persistent_workers=True
)

scaler = torch.amp.GradScaler('cuda')
print(f'Train: {n_train} | Val: {n_val} | Batches/epoch: {len(train_loader)}')


Starting from scratch.
Using 2 GPUs
FLAMEOnlyDataset: 19582 sequences found.
Train: 17624 | Val: 1958 | Batches/epoch: 1102


In [9]:
def get_kl_free_bits(epoch):
    return max(0.1, 0.5 * (1.0 - epoch / 100.0))


def kl_loss_fn(mean, logvar, free_bits=0.5):
    kl_per_dim = -0.5 * (1.0 + logvar - mean.pow(2) - logvar.exp())
    kl_per_dim = kl_per_dim.clamp(min=free_bits)
    return kl_per_dim.sum(dim=-1).mean()


def get_kl_beta(epoch):
    if epoch >= KL_WARMUP_EPOCHS:
        return KL_BETA_TARGET
    return KL_BETA_TARGET * (epoch / KL_WARMUP_EPOCHS)


In [10]:
MAX_HOURS      = 11
max_time_secs  = MAX_HOURS * 3600
start_time     = time.time()
time_limit_hit = False

hdr = f"| {'Epoch':^7} | {'Train':^10} | {'Val':^10} | {'Recon':^10} | {'KL':^10} | {'β':^8} |"
sep = '-' * len(hdr)
print(sep); print(hdr); print(sep)

total_steps = (EPOCHS - start_epoch) * len(train_loader)
pbar = tqdm(total=total_steps, desc='KL-VAE')

for epoch in range(start_epoch, EPOCHS):
    weights = loss_scheduler.get_weights(epoch)
    base_crit = criterion.module if hasattr(criterion, 'module') else criterion
    base_crit.update_weights(weights)
    if (epoch % 10) == 0:
        print(f"  [Weights ep{epoch}] " +
              " | ".join(f"{k}={v:.1f}" for k, v in weights.items()))

    kl_beta      = get_kl_beta(epoch)
    kl_free_bits = get_kl_free_bits(epoch)

    klvae.train()
    recon_sum = kl_sum = 0.0
    n_batches = 0

    for batch in train_loader:
        if time.time() - start_time > max_time_secs:
            tqdm.write('\n[Time Limit] Stopping safely.')
            time_limit_hit = True
            break

        gt_flame = batch['flame'].to(device)
        mask     = batch['mask'].to(device)
        gt_shape = batch['shape'].to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type='cuda', dtype=torch.float16):
            recon, mean, logvar = klvae(gt_flame, mask)
            loss_recon = criterion(recon, gt_flame, mask, gt_shape)
            loss_kl    = kl_loss_fn(mean, logvar, free_bits=kl_free_bits)
            total_loss = loss_recon + kl_beta * loss_kl

        scaler.scale(total_loss.mean()).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(klvae.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        recon_sum += loss_recon.mean().item()
        kl_sum    += loss_kl.mean().item()
        n_batches += 1
        pbar.update(1)
        pbar.set_postfix({
            'ep':    f'{epoch+1}/{EPOCHS}',
            'recon': f'{loss_recon.mean().item():.4f}',
            'kl':    f'{loss_kl.mean().item():.2f}',
            'β':     f'{kl_beta:.2e}',
            'fb':    f'{kl_free_bits:.2f}',
        })

    if time_limit_hit:
        break

    scheduler.step()
    avg_recon = recon_sum / max(n_batches, 1)
    avg_kl    = kl_sum    / max(n_batches, 1)
    avg_train = avg_recon + kl_beta * avg_kl

    if (epoch + 1) % 5 == 0 or time_limit_hit:
        klvae.eval()
        val_recon_sum = val_kl_sum = 0.0
        val_batches   = 0

        with torch.no_grad():
            for vb in val_loader:
                vgt   = vb['flame'].to(device)
                vmask = vb['mask'].to(device)
                vshp  = vb['shape'].to(device)
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    vrecon, vmean, vlogvar = klvae(vgt, vmask)
                    vl_recon = criterion(vrecon, vgt, vmask, vshp)
                    vl_kl    = kl_loss_fn(vmean, vlogvar, free_bits=kl_free_bits)
                val_recon_sum += vl_recon.mean().item()
                val_kl_sum    += vl_kl.mean().item()
                val_batches   += 1

        avg_val_recon = val_recon_sum / max(val_batches, 1)
        avg_val_kl    = val_kl_sum    / max(val_batches, 1)
        avg_val       = avg_val_recon + kl_beta * avg_val_kl

        if avg_val_kl < 2.0 and epoch > KL_WARMUP_EPOCHS:
            tqdm.write(f"KL Collapse! KL dropped to {avg_val_kl:.2f}. Scaling back beta.")
            KL_BETA_TARGET *= 0.8 # Dynamic reduction to save the latent space

        base = klvae.module if hasattr(klvae, 'module') else klvae
        ckpt_path = f'/kaggle/working/klvae_ep{epoch+1}.pth'
        torch.save({
            'epoch':           epoch + 1,
            'klvae_state':     base.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'best_val_loss':   best_val_loss,
            'latent_dim':      base.latent_dim,
        }, ckpt_path)

        tag = ''
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            best_path = '/kaggle/working/klvae_best.pth'
            torch.save({
                'epoch':           epoch + 1,
                'klvae_state':     base.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'scheduler_state': scheduler.state_dict(),
                'best_val_loss':   best_val_loss,
                'latent_dim':      base.latent_dim,
            }, best_path)
            tag = ' ★ best'

        row = (f"| {epoch+1:^7} | {avg_train:^10.4f} | {avg_val:^10.4f} "
               f"| {avg_val_recon:^10.4f} | {avg_val_kl:^10.2f} | {kl_beta:^8.2e} |{tag}")
        print(row)
    else:
        row = (f"| {epoch+1:^7} | {avg_train:^10.4f} | {'—':^10} "
               f"| {'—':^10} | {'—':^10} | {kl_beta:^8.2e} |")
        print(row)

pbar.close()
print(sep)
print(f'Training complete. Best val loss: {best_val_loss:.4f}')


--------------------------------------------------------------------------
|  Epoch  |   Train    |    Val     |   Recon    |     KL     |    β     |
--------------------------------------------------------------------------


KL-VAE:   0%|          | 0/220400 [00:00<?, ?it/s]

  [Weights ep0] w_recon=5.0 | w_motion=0.1 | w_jaw=5.0 | w_mouth=1000.0 | w_lower=500.0 | w_upper=1500.0


KL-VAE:   0%|          | 1102/220400 [06:06<19:46:14,  3.08it/s, ep=1/200, recon=0.0065, kl=65.66, β=0.00e+00, fb=0.50]

|    1    |   0.0386   |     —      |     —      |     —      | 0.00e+00 |


...

|   103   |   0.0016   |     —      |     —      |     —      | 1.00e-04 |


KL-VAE:  52%|█████▏    | 114608/220400 [10:53:46<8:17:20,  3.55it/s, ep=104/200, recon=0.0005, kl=11.25, β=1.00e-04, fb=0.10]

|   104   |   0.0017   |     —      |     —      |     —      | 1.00e-04 |


KL-VAE:  52%|█████▏    | 115703/220400 [11:00:00<9:57:13,  2.92it/s, ep=105/200, recon=0.0007, kl=10.24, β=1.00e-04, fb=0.10]


[Time Limit] Stopping safely.
--------------------------------------------------------------------------
Training complete. Best val loss: 0.0009


In [11]:
@torch.no_grad()
def encode_to_latent(flame_tensor, mask=None):
    base = klvae.module if hasattr(klvae, 'module') else klvae
    base.eval()
    squeeze = flame_tensor.dim() == 2
    if squeeze:
        flame_tensor = flame_tensor.unsqueeze(0)
    flame_tensor = flame_tensor.to(device)
    latent = base.encode(flame_tensor, mask)
    return latent.squeeze(0) if squeeze else latent


@torch.no_grad()
def decode_from_latent(z):
    base = klvae.module if hasattr(klvae, 'module') else klvae
    base.eval()
    squeeze = z.dim() == 2
    if squeeze:
        z = z.unsqueeze(0)
    out = base.decoder(z.to(device))
    return out.squeeze(0) if squeeze else out


klvae.eval()
sb    = next(iter(val_loader))
gt_s  = sb['flame'][:1].to(device)
msk_s = sb['mask'][:1].to(device)

base_kl = klvae.module if hasattr(klvae, 'module') else klvae
recon_s, mean_s, logvar_s = base_kl(gt_s, msk_s)

vlen = msk_s[0].sum().item()
mse  = F.mse_loss(recon_s[0, :vlen], gt_s[0, :vlen]).item()
kl_s = kl_loss_fn(mean_s, logvar_s).item()

print(f'Sanity check')
print(f'  Param-space MSE : {mse:.6f}')
print(f'  KL (nats/dim)   : {kl_s:.4f}')
print(f'  Seq length      : {int(vlen)} frames')
print(f'  Latent shape    : {mean_s.shape}  (B, T, {LATENT_DIM})')
print()

Sanity check
  Param-space MSE : 0.140909
  KL (nats/dim)   : 17.4019
  Seq length      : 42 frames
  Latent shape    : torch.Size([1, 188, 32])  (B, T, 32)



In [12]:
@torch.no_grad()
def klvae_reconstruction(flame_path, device):
    gt_dict = torch.load(flame_path, weights_only=True, map_location=device)
    
    expressions = gt_dict['exp']
    jaw_pose    = gt_dict['pose'][:, 3:6]
    gt_flame    = torch.cat([expressions, jaw_pose], dim=-1)[:, :53].unsqueeze(0).to(device)
    
    latents = encode_to_latent(gt_flame)
    reconstructed_flame = decode_from_latent(latents)
    
    return reconstructed_flame.squeeze(0)

In [13]:
def export_prediction_to_video(predicted_tensor, video_filename, audio_path, flame_model, device, sample_flame_path, fps=25):
    if hasattr(flame_model, 'module'):
        flame_model = flame_model.module

    if predicted_tensor.dim() == 3:
        predicted_tensor = predicted_tensor.squeeze(0)
        
    gt_dict = torch.load(sample_flame_path, weights_only=True, map_location=device)
    expressions = gt_dict['exp']
    jaw_pose = gt_dict['pose'][:, 3:6]
    gt_tensor = torch.cat([expressions, jaw_pose], dim=-1)[:, :53]

    audio_clip = AudioFileClip(audio_path)
    audio_duration = audio_clip.duration
    target_frame_count = int(np.ceil(audio_duration * fps))

    def adjust_tensor(tensor):
        current_frames = tensor.shape[0]
        if current_frames < target_frame_count:
            padding = tensor[-1:].expand(target_frame_count - current_frames, -1)
            tensor = torch.cat([tensor, padding], dim=0)
        elif current_frames > target_frame_count:
            tensor = tensor[:target_frame_count]
        return tensor

    if predicted_tensor.shape[0] < 5:
        return

    predicted_tensor = adjust_tensor(predicted_tensor)
    gt_tensor = adjust_tensor(gt_tensor)
    T = target_frame_count
    base_shape = gt_dict['shape'][0:1].to(device)

    def get_verts(tensor):
        all_verts = []
        chunk_size = 500
        with torch.no_grad():
            for i in range(0, T, chunk_size):
                chunk = tensor[i:i+chunk_size].to(device)
                fixed_shape = base_shape.expand(chunk.shape[0], -1)
                pose = torch.zeros((chunk.shape[0], 6), device=chunk.device, dtype=chunk.dtype)
                pose[:, 3:6] = chunk[:, 50:53]
                flame_output = flame_model(
                    shape_params=fixed_shape,
                    expression_params=chunk[:, :50],
                    pose_params=pose
                )
                all_verts.append(flame_output[0].cpu().numpy())
        return np.concatenate(all_verts, axis=0)

    pred_verts = get_verts(predicted_tensor)
    gt_verts = get_verts(gt_tensor)
    faces = flame_model.faces_tensor.cpu().numpy()

    renderer = pyrender.OffscreenRenderer(viewport_width=512, viewport_height=512)
    camera = pyrender.PerspectiveCamera(yfov=np.pi / 3.0, aspectRatio=1.0)
    camera_pose = np.array([
        [1.0,  0.0,  0.0,  0.0],
        [0.0,  1.0,  0.0,  0.0],
        [0.0,  0.0,  1.0,  0.3], 
        [0.0,  0.0,  0.0,  1.0]
    ])
    
    light = pyrender.DirectionalLight(color=[1.0, 1.0, 1.0], intensity=2.0)
    temp_video = "temp_silent.mp4"
    writer = imageio.get_writer(temp_video, fps=fps)
    
    for i in range(T):
        gt_mesh = trimesh.Trimesh(vertices=gt_verts[i], faces=faces)
        gt_mat = pyrender.MetallicRoughnessMaterial(
            metallicFactor=0.1, alphaMode='OPAQUE', baseColorFactor=(0.6, 0.7, 0.6, 1.0)
        )
        scene_gt = pyrender.Scene(bg_color=[0.0, 0.0, 0.0, 0.0])
        scene_gt.add(pyrender.Mesh.from_trimesh(gt_mesh, material=gt_mat))
        scene_gt.add(camera, pose=camera_pose)
        scene_gt.add(light, pose=camera_pose)
        color_gt, _ = renderer.render(scene_gt)
        
        pred_mesh = trimesh.Trimesh(vertices=pred_verts[i], faces=faces)
        pred_mat = pyrender.MetallicRoughnessMaterial(
            metallicFactor=0.1, alphaMode='OPAQUE', baseColorFactor=(0.7, 0.6, 0.6, 1.0)
        )
        scene_pred = pyrender.Scene(bg_color=[0.0, 0.0, 0.0, 0.0])
        scene_pred.add(pyrender.Mesh.from_trimesh(pred_mesh, material=pred_mat))
        scene_pred.add(camera, pose=camera_pose)
        scene_pred.add(light, pose=camera_pose)
        color_pred, _ = renderer.render(scene_pred)
        
        color_concat = np.concatenate((color_gt, color_pred), axis=1)
        writer.append_data(color_concat)
        
    writer.close()
    renderer.delete()
    
    video_clip = VideoFileClip(temp_video)
    final_clip = video_clip.set_audio(audio_clip)
    final_clip.write_videofile(
        video_filename, codec="libx264", audio_codec="aac", fps=fps, verbose=False, logger=None
    )
    os.remove(temp_video)

In [14]:
def generate_inference_videos(id_list, lang, flame_model, device):
    base_flame_model = flame_model.module if hasattr(flame_model, 'module') else flame_model
    for item_id in id_list:
        sample_wav_path = f"/kaggle/input/datasets/jorx04/fully-collected-dataset/audio/{lang}/{item_id}.wav"
        sample_flame_path = f"/kaggle/input/datasets/jorx04/fully-collected-dataset/flame/{lang}/{item_id}_flame_params.pt"
        output_vid = f"/kaggle/working/vid_recon_{item_id}.mp4"

        if os.path.exists(sample_flame_path) and os.path.exists(sample_wav_path):
            t0 = time.time()

            reconstructed_sequence = klvae_reconstruction(sample_flame_path, device)

            export_prediction_to_video(
                predicted_tensor=reconstructed_sequence,
                video_filename=output_vid,
                audio_path=sample_wav_path,
                flame_model=base_flame_model,
                device=device,
                sample_flame_path=sample_flame_path
            )

            elapsed = time.time() - t0
            print(f"[{item_id}] done in {elapsed:.1f}s")
        else:
            print(f"[{item_id}] skipped — file not found")

In [15]:
ara_files = []
eng_files = []

for i in range(len(val_dataset)):
    if len(ara_files) >= 3 and len(eng_files) >= 3:
        break

    orig_idx = val_dataset.indices[i]
    
    f_path = val_dataset.dataset.samples[orig_idx]
    
    is_ara = '/ara/' in f_path
    is_eng = '/eng/' in f_path
    
    if is_ara and len(ara_files) >= 3:
        continue
    if is_eng and len(eng_files) >= 3:
        continue

    flame_seq, shape_param = val_dataset[i]
    
    if flame_seq.shape[0] >= 150:
        base_name = os.path.basename(f_path).replace("_flame_params.pt", "")
        if is_ara:
            ara_files.append(base_name)
        elif is_eng:
            eng_files.append(base_name)

print("Arabic Files (>= 150 frames):", ara_files)
print("English Files (>= 150 frames):", eng_files)

Arabic Files (>= 150 frames): ['5sabyHgNs5M_6', 'znzkPpWwJBk_438', '9uELlqKSTLs_9']
English Files (>= 150 frames): ['SuzYgORx3Es_9', 'uNHaBIvEzeo_7', 'jQTsUhVFbYA_18']


In [16]:
ara_files.extend(["0YJdaGwpUHA_0", "0YJdaGwpUHA_1"])
eng_files.extend(["-WB_SGqeJo8_15", "-m8tHXm8D8w_18"])

In [17]:
generate_inference_videos(ara_files, "ara", flame_model, device)
generate_inference_videos(eng_files, "eng", flame_model, device)

[5sabyHgNs5M_6] done in 31.0s
[znzkPpWwJBk_438] done in 21.2s
[9uELlqKSTLs_9] done in 19.0s
[0YJdaGwpUHA_0] done in 20.6s
[0YJdaGwpUHA_1] done in 19.0s
[SuzYgORx3Es_9] done in 17.2s
[uNHaBIvEzeo_7] done in 16.7s
[jQTsUhVFbYA_18] done in 34.3s
[-WB_SGqeJo8_15] done in 20.6s
[-m8tHXm8D8w_18] done in 16.7s


In [18]:
!rm -r /kaggle/working/spectre